In [ ]:
"""
Morris-Lecar Model: Phase Portrait, Nullclines, and Trajectory Simulation.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import fsolve

# Глобальная настройка шрифтов для презентабельного вида графиков
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 14,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
})

# Константы биологической модели Морриса-Лекара
PARAMS = {
    'C': 20.0, 'g_leak': 0.20, 'V_leak': -50.0,
    'g_Ca': 4.4, 'V_Ca': 100.0, 'g_K': 8.0, 'V_K': -70.0,
    'V1': -1.0, 'V2': 15.0, 'V3': 2.0, 'V4': 30.0, 'phi': 0.05
}

def inf_m(V):
    return 1 / (1 + np.exp((PARAMS['V1'] - V) / PARAMS['V2']))

def inf_n(V):
    return 1 / (1 + np.exp((PARAMS['V3'] - V) / PARAMS['V4']))

def inf_Veq(V, I_hold):
    """Стационарная функция для нахождения точек покоя системы (dV/dt = 0)."""
    m_inf = inf_m(V)
    n_inf = inf_n(V)
    I_leak = PARAMS['g_leak'] * (V - PARAMS['V_leak'])
    I_Ca = PARAMS['g_Ca'] * m_inf * (V - PARAMS['V_Ca'])
    I_K = PARAMS['g_K'] * n_inf * (V - PARAMS['V_K'])
    return (-I_leak - I_Ca - I_K + I_hold) / PARAMS['C']

def DBolt(V5, V, k):
    """Производная функции Больцмана."""
    return np.exp((V5 - V) / k) / (k * (1 + np.exp((V5 - V) / k))**2)

def trJ(V):
    """След матрицы Якобиана системы в точке V."""
    m_inf = inf_m(V)
    n_inf = inf_n(V)
    Dm_inf = DBolt(PARAMS['V1'], V, PARAMS['V2'])
    return -PARAMS['phi'] - (PARAMS['g_leak'] + PARAMS['g_K'] * n_inf +
             PARAMS['g_Ca'] * (m_inf + Dm_inf * (V - PARAMS['V_Ca']))) / PARAMS['C']

def detJ(V):
    """Определитель матрицы Якобиана системы в точке V."""
    m_inf = inf_m(V)
    n_inf = inf_n(V)
    Dm_inf = DBolt(PARAMS['V1'], V, PARAMS['V2'])
    Dn_inf = DBolt(PARAMS['V3'], V, PARAMS['V4'])
    return (PARAMS['g_leak'] + PARAMS['g_K'] * (n_inf + Dn_inf * (V - PARAMS['V_K'])) +
            PARAMS['g_Ca'] * (m_inf + Dm_inf * (V - PARAMS['V_Ca']))) / PARAMS['C'] * PARAMS['phi']

def morris_lecar(t, y, I):
    """Правая часть системы обыкновенных дифференциальных уравнений."""
    V, n = y
    I_leak = PARAMS['g_leak'] * (V - PARAMS['V_leak'])
    I_Ca = PARAMS['g_Ca'] * inf_m(V) * (V - PARAMS['V_Ca'])
    I_K = PARAMS['g_K'] * n * (V - PARAMS['V_K'])

    dVdt = (-I_leak - I_Ca - I_K + I) / PARAMS['C']
    dn_dt = PARAMS['phi'] * (inf_n(V) - n)
    return [dVdt, dn_dt]

def plot_phase_plane(I=8.8):
    print(f"Расчет фазовой плоскости для внешнего тока I = {I} мкА/см²...")

    # Настройка расчетной сетки для изоклин
    V_range = np.linspace(-85, 30, 200)
    n_range = np.linspace(0, 0.6, 200)
    V_grid, n_grid = np.meshgrid(V_range, n_range)

    # Функция для вычисления нулевого уровня V-изоклины
    def v_nullcline_mesh(V, n, I_hold):
        m_inf = inf_m(V)
        I_leak = PARAMS['g_leak'] * (V - PARAMS['V_leak'])
        I_Ca = PARAMS['g_Ca'] * m_inf * (V - PARAMS['V_Ca'])
        I_K = PARAMS['g_K'] * n * (V - PARAMS['V_K'])
        return -I_leak - I_Ca - I_K + I_hold

    F_V = v_nullcline_mesh(V_grid, n_grid, I)

    # Определение координат особой точки (равновесия)
    V_eq = fsolve(inf_Veq, -20.0, args=(I,))[0]
    n_eq = inf_n(V_eq)

    print("Интегрирование фазовой траектории (старт строго с n-изоклины)...")
    V_start_traj = -60.0
    n_start_traj = inf_n(V_start_traj)

    sol_traj = solve_ivp(
        lambda t, y: morris_lecar(t, y, I),
        t_span=[0, 2000],
        y0=[V_start_traj, n_start_traj],
        t_eval=np.linspace(0, 2000, 10000),
        method='LSODA',
        rtol=1e-6
    )

    # --- ГРАФИК 1: ФАЗОВЫЙ ПОРТРЕТ ---
    plt.figure(figsize=(11, 8))

    # Изоклина V=0 (Красная)
    plt.contour(V_grid, n_grid, F_V, levels=[0], colors='red', linewidths=2.5)
    # Костыль для отображения легенды contour
    plt.plot([], [], color='red', linewidth=2.5, label='Изоклина $\dot{V} = 0$')

    # Изоклина n=0 (Голубая)
    plt.plot(V_range, inf_n(V_range), color='cyan', linewidth=2.5, label='Изоклина $\dot{n} = 0$')

    # Интегральная кривая (Черная траектория)
    plt.plot(sol_traj.y[0], sol_traj.y[1], 'k-', linewidth=3.5, label='Фазовая траектория $(V(t), n(t))$')

    # Точки старта и устойчивого фокуса/узла
    plt.plot(V_start_traj, n_start_traj, 'go', markersize=10, label='Точка старта (на изоклине $n$)')
    plt.plot(V_eq, n_eq, 'mo', markersize=14, markeredgecolor='white', label=f'Равновесие ({V_eq:.1f}, {n_eq:.2f})')

    plt.xlabel('$V$ (мВ)', fontsize=14)
    plt.ylabel('$n$', fontsize=14)
    plt.title(f'Фазовый портрет модели Морриса-Лекара ($I = {I}$ мкА/см²)', fontsize=16, pad=15)
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3, linestyle=':')
    plt.xlim(-70, 30)
    plt.ylim(0, 0.6)
    plt.tight_layout()
    plt.show()

    # Вывод инвариантов линеаризации в консоль
    print(f"\nПараметры устойчивости для V_eq = {V_eq:.2f} мВ:")
    print(f"След Якобиана tr(J) = {trJ(V_eq):.4f}")
    print(f"Определитель Якобиана detJ = {detJ(V_eq):.4f}")

    # --- ГРАФИК 2: ВРЕМЕННАЯ РАЗВЕРТКА ПОТЕНЦИАЛА ---
    plt.figure(figsize=(11, 4))
    plt.plot(sol_traj.t, sol_traj.y[0], 'r-', linewidth=2, label='$V(t)$')
    plt.title('Временная динамика потенциала мембраны', fontsize=14)
    plt.xlabel('Время (мс)')
    plt.ylabel('$V$ (мВ)')
    plt.grid(True, alpha=0.3, linestyle=':')
    plt.xlim(0, 2000)
    plt.legend()
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    plot_phase_plane(I=8.8)